In [1]:
import os
from dotenv import load_dotenv
import xarray as xr

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import date

import analysis_utils
import isku_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'isku_utils' from '/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py'>

In [2]:
load_dotenv()
#DATA_DIR = os.environ["DATA_DIR"]
# Baseline period for Impact
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")
# Define Forecast Months
FC_MONTHS = [9, 10, 11, 12, 1, 2]
FC_PERIOD = slice("2026-09-01", "2027-02-28")

config = analysis_utils.ImpactConfig( version = "v260910",
                                     baseline_period=BASELINE_PERIOD, 
                                     polygons_path= os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
                                     socioeconomics_path= os.environ["POREALLAS_SOCIOECONOMICS_URI"],
                                     rate=False, 
                                     months = FC_MONTHS,
                                     hotonly = "hotonly", 
                                     dims = ['number', 'sample'])

# Define Forecast
EFFECTS_URI = "/home/emily_zuetell/projects/poreallas/data/v20260909_effects_with_betas.zarr" # Can be any effects datatree

In [3]:
# Projection Effects
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)

In [4]:
### Log baseline period and impact calculation
rate_l = "rate" if config.rate else "total"
baseline_tag = analysis_utils._baseline_tag(config.baseline_period)

In [5]:
# Compute impact: forecast - baseline
impact = analysis_utils.compute_impact(
    effect.chunk({dim: -1 for dim in config.dims}),
    config,
    ensemble=True,
)
# Use only the defined 6-months
impact = impact.sel(month=config.months)

In [6]:
# Aggregate Impact Regions to group_level
group_level = 'ISO' #IR: Impact region, # ADM1: State level, # ISO: Country level
# Use a subset of ensemble members for quick testing (.sel(number = ...))
#impact = impact.sel(number = [0, 1, 2, 3, 4])
impact, merge_key, base_cols = analysis_utils.aggregate_impact(impact, config, group_level)

In [8]:
def crossed_sem(da, a="number", b="sample", include_resid = False):
    # Crossed standard error
    # Size of dims (number, sample)
    na, nb = da.sizes[a], 100
    # grand mean
    grand = da.mean((a, b))
    m_a = da.mean(b)                     # per-member means (over samples)
    m_b = da.mean(a)                     # per-sample means (over members)

    resid = da - m_a - m_b + grand       # interaction/residual term
    if include_resid:
        var_e = (resid**2).sum((a, b)) / ((na - 1) * (nb - 1))
    else:
        var_e = 0
    
    var_mean = (m_a.var(a, ddof=1) / na
                + m_b.var(b, ddof=1) / nb
                - var_e / (na * nb))
    return grand, np.sqrt(var_mean.clip(min=0))

In [9]:
mean, se = crossed_sem(impact.sum(dim = 'month'), a="number", b="sample", include_resid = True)
mean, se_no_resid = crossed_sem(impact.sum(dim = 'month'), a="number", b="sample", include_resid = False)

In [10]:
no_resid = se_no_resid.to_dataframe()['age_weighted_impact']

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


In [11]:
df = se.to_dataframe()

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


In [12]:
df['no_resid'] = no_resid

In [13]:
df = df.rename(columns = {'age_weighted_impact': 'resid'})

In [14]:
df

,year,resid,no_resid
ISO,,,
ABW,2026,0.264177,0.264194
AFG,2026,20.323694,20.332949
AGO,2026,107.783940,107.792203
AIA,2026,0.006889,0.006889
ALA,2026,0.000802,0.000803
...,...,...,...
WSM,2026,0.522420,0.522479
YEM,2026,104.568377,104.572036
ZAF,2026,138.877123,138.904005


In [43]:
df.to_csv('crossed_se_hotonly_6mo.csv')

In [52]:
#Compute stats in xarray from dims in config.dims
stat_cols = ["median", "p17", "p83", "likely_range_IPCC", "mean", "std", "min", "max", "p10", "p90"]
_polygons_impact = analysis_utils.dataset_to_dataframe(analysis_utils.compute_stats(impact, dim=config.dims))

In [54]:
filename_template="{version}_{hotonly}_{scope}_{rate_l}_{stat_scope}_{group_level}_{baseline}_{cleaned}.csv"
# "regional_monthly" 
_polygons_impact = analysis_utils.dataset_to_dataframe(analysis_utils.compute_stats(impact, dim=config.dims))
wide = _polygons_impact.pivot(
    index=base_cols,
    columns="month", values=stat_cols,
)
wide.columns = [f"month {m} {stat}" for stat, m in wide.columns]
stat_col_names = wide.columns.difference(base_cols)
wide = wide.reset_index()
wide_rounded = analysis_utils.round_output(wide)
wide.to_csv(
    filename_template.format(
        version=config.version,
        hotonly=config.hotonly,
        rate_l=rate_l,
        scope="monthly",
        stat_scope="",
        group_level=group_level,
        baseline=baseline_tag,
        cleaned = 'raw'
    ),
    index=False,
)
wide_rounded.to_csv(
    filename_template.format(
        version=config.version,
        hotonly=config.hotonly,
        rate_l=rate_l,
        scope="monthly",
        stat_scope="",
        group_level=group_level,
        baseline=baseline_tag,
        cleaned = 'rounded'
    ),
    index=False,
)
